# Plots

In [1]:
import datetime
import os

import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
sns.set_theme(style="whitegrid", palette="deep")
task_palette = sns.color_palette("deep")
plt.rcParams["text.usetex"] = True
plt.rcParams.update({"figure.titlesize": "small"})
plt.rcParams.update({"axes.titlesize": "small"})
plt.rcParams.update({"axes.labelsize": "small"})
plt.rcParams.update({"ytick.labelsize": "small"})
plt.rcParams.update({"xtick.labelsize": "small"})
plt.rcParams.update({"legend.fontsize": "small"})

## Antichain Test

In [ ]:
path = "./results/results-mcs_scale_bfs-2026-04-15_20-29-06.csv"
df = pd.read_csv(path, sep=";")
df

In [ ]:
df["duration_s"] = df["duration_ns"] / 1e9
df["search_type"] = df["use_idlesim"].apply(lambda x: "ACBFS" if x else "BFS")
df = df.dropna(how="any")

In [ ]:
s = df.shape[0] * [True]
df_plot = df.loc[s]
df_piv = df_plot.copy().pivot(
    index="taskset_position",
    columns="search_type",
    values=["duration_s", "visited_count", "max_period", "nbt"],
)
df_piv.columns = list(map(lambda x: "_".join(x), df_piv.columns))
df_piv = df_piv.dropna(how="any")

In [ ]:
df_piv["Number of tasks"] = df_piv["nbt_BFS"].astype(int)
df_piv["Period max"] = df_piv["max_period_ACBFS"].astype(int)
df_piv["n"] = df_piv["nbt_BFS"].astype(int)
df_piv["T max"] = df_piv["max_period_ACBFS"].astype(int)

In [ ]:
f, ax = plt.subplots(figsize=(6, 3))

sns.scatterplot(
    data=df_piv,
    x="duration_s_BFS",
    y="duration_s_ACBFS",
    style="T max",
    hue="n",
    ax=ax,
    palette=task_palette,
)
ax.set(xscale="log", yscale="log")
# ax.axis("equal")

lims = [
    10**-5.1,
    np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

# now plot both limits against eachother
ax.plot(lims, lims, "k--", alpha=0.75, zorder=1)

ax.set(xlim=lims, ylim=lims)

ax.set_xlabel("BFS")
ax.set_ylabel("ACBFS")
ax.set_title("Execution time (s)")

# sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1.1))

f.suptitle(r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, U^{*} = 0.5$", y=1.02)

h, l = ax.get_legend_handles_labels()
ax.legend_.remove()
leg = [
    "$n$",
    "2",
    "3",
    "4",
    "5",
    "6",
    "$T^{\mathsf{max}}$",
    "10",
    "15",
    "20",
    "25",
    "30",
]
f.legend(h, leg, ncol=2, loc="upper left", bbox_to_anchor=(0.15, 0.85), framealpha=1)


x = np.log10(df_piv.sort_values(by="duration_s_BFS")["duration_s_BFS"])
y = np.log10(df_piv.sort_values(by="duration_s_BFS")["duration_s_ACBFS"])
m, b = np.polyfit(x, y, 1)
x_reg = np.linspace(lims, 100)
x_reg = np.log10(x_reg)
plt.plot(10**x_reg, 10 ** (m * x_reg + b), color="k", ls=":")

In [ ]:
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_bfs_acbfs.pdf",
    bbox_inches="tight",
)
f.savefig(
    "./figs/bfs_acbfs.pdf",
    bbox_inches="tight",
)

## Scalability Test

In [ ]:
path = "./results/results-mcs_scale01-2026-04-15_20-29-48.csv"

df_sp = pd.read_csv(path, sep=";")
df_sp

In [ ]:
df_sp["taskset_file"].value_counts()

In [ ]:
df_sp.groupby("taskset_file")["taskset_position"].max()

In [ ]:
df_sp["duration_s"] = df_sp["duration_ns"] / 1e9

In [ ]:
df_sp["timeout_int"] = df_sp["timeout"].astype(int)
df_sp.groupby(["nbt"])["timeout_int"].agg(["count", "sum"])

In [ ]:
df_sp.groupby("nbt")["duration_s"].describe()

In [ ]:
# f, axs = plt.subplots(2, 1, figsize=(3.5*52/98*2, 3.5), gridspec_kw={"hspace": 0.25})
f, axs = plt.subplots(
    2, 1, figsize=(3.2 / 1.05, 3.2), gridspec_kw={"hspace": 0.25}, sharey=True
)


ax1 = axs[0]
ax2 = ax1.twinx()

s = df_sp["taskset_file"].str.contains("task")
dimension = "nbt"

# s = df_sp["taskset_file"].str.contains("period")
# dimension = "max_period"

s &= df_sp["use_case"] == "ACBFS, no oracle"


n_xp = 50

x_val = np.sort(df_sp.loc[s, dimension].unique())
x_val = list(map(int, x_val))
y_val_line = []
y_val_bar = []
for x in x_val:
    y_line = df_sp.loc[s & (df_sp[dimension] == x), "duration_s"].mean()
    y_val_line.append(y_line)

    y_bar = df_sp.loc[s & (df_sp[dimension] == x), "timeout_int"].sum()
    y_bar += n_xp - df_sp.loc[s & (df_sp[dimension] == x), "timeout_int"].count()
    y_val_bar.append(y_bar)

sns.lineplot(
    x=x_val,
    y=y_val_line,
    ax=ax1,
    marker="o",
    ms=9,
    estimator="median",
)
ax1.set(yscale="log")

ax2.bar(
    x=x_val,
    height=y_val_bar,
    color="w",
    edgecolor=task_palette[1],
    hatch="//",
    alpha=1,
    width=(x_val[1] - x_val[0]) * 0.8,
)

ax2.grid(False)
ax2.set_ylim(0, n_xp)


ax1.set_xlabel("Number of tasks ($n$), $T^{\mathsf{max}} = 30$")
ax1.set_ylabel("Execution time (s)")
ax2.set_ylabel("Timeouts")
ax1.minorticks_off()

ax1.xaxis.tick_top()
ax1.xaxis.set_label_position("top")
ax1.xaxis.set_ticks_position("none")
ax1.yaxis.set_ticks_position("none")
ax2.yaxis.set_ticks_position("none")
ax1.tick_params(axis="x", which="major", pad=0)


ax1 = axs[1]
ax2 = ax1.twinx()

# s = df_sp["taskset_file"].str.contains("task")
# dimension = "nbt"

s = df_sp["taskset_file"].str.contains("period")
dimension = "max_period"

s &= df_sp["use_case"] == "ACBFS, no oracle"


x_val = np.sort(df_sp.loc[s, dimension].unique())
x_val = list(map(int, x_val))
y_val_line = []
y_val_bar = []
for x in x_val:
    y_line = df_sp.loc[s & (df_sp[dimension] == x), "duration_s"].mean()
    y_val_line.append(y_line)

    y_bar = df_sp.loc[s & (df_sp[dimension] == x), "timeout_int"].sum()
    y_bar += n_xp - df_sp.loc[s & (df_sp[dimension] == x), "timeout_int"].count()
    y_val_bar.append(y_bar)

sns.lineplot(
    x=x_val,
    y=y_val_line,
    ax=ax1,
    marker="o",
    ms=9,
    estimator="median",
)
ax1.set(yscale="log")

ax2.bar(
    x=x_val,
    height=y_val_bar,
    color="w",
    edgecolor=task_palette[1],
    hatch="//",
    alpha=1,
    width=(x_val[1] - x_val[0]) * 0.8,
)

ax2.grid(False)
ax2.set_ylim(0, n_xp)


ax1.set_xlabel("Period max ($T^{\mathsf{max}}$), $n = 5$")
ax1.set_ylabel("Execution time (s)", labelpad=10)
ax2.set_ylabel("Timeouts")
ax1.minorticks_off()
ax1.yaxis.set_ticks_position("none")
ax2.yaxis.set_ticks_position("none")

f.suptitle(
    r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, U^{*} = 0.5$",
    # x=0.55,
    # y=0.96,
    y=1.05,
)

blue_line = mlines.Line2D(
    [],
    [],
    color=task_palette[0],
    marker="o",
    linestyle="-",
    markeredgecolor="w",
    label="Execution time (s)",
    ms=9,
)

orange_box = mpatches.Patch(
    hatch="//", label="Timeouts", edgecolor=task_palette[1], facecolor="white"
)

new_handles = [blue_line, orange_box]


ax1.legend(
    handles=new_handles,
    loc="upper center",
    ncol=2,
    framealpha=0,
    columnspacing=0.8,
    bbox_to_anchor=(0.5, 1.28),
)

ax1.get_ylim()


f.tight_layout()

In [ ]:
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_tmax_ntask.pdf",
    bbox_inches="tight",
)
f.savefig(
    "./figs/tmax_ntask.pdf",
    bbox_inches="tight",
)

## Schedulability Test
### Input data

In [36]:
from pathlib import Path

# Load baseline results from a single CSV file or from every CSV under a directory.
# When multiple files are loaded, later files can overwrite use cases already present.
path = Path("./results/0526_02")
if path.is_dir():
    csv_paths = sorted(path.glob("*.csv"))
else:
    csv_paths = [path]

if not csv_paths:
    raise FileNotFoundError(f"No CSV files found at {path}")

# Start from the first file, then merge the remaining ones one by one.
df_sch = pd.read_csv(csv_paths[0], sep=";")
for patch_path in csv_paths[1:]:
    df_patch = pd.read_csv(patch_path, sep=";")
    new_use_cases = set(df_patch["use_case"].unique())
    overwritten_use_cases = set(df_sch["use_case"].unique()).intersection(new_use_cases)
    if overwritten_use_cases:
        print(f"Overwriting use cases: {overwritten_use_cases}")
        df_sch = df_sch[~df_sch["use_case"].isin(overwritten_use_cases)]
    df_sch = pd.concat([df_sch, df_patch], ignore_index=True)

df_sch

,safe_oracles,unsafe_oracles,use_case,scheduler,search_algorithms,taskset_file,taskset_position,U,Uv,nbt,...,is_safe_1,automaton_depth_1,visited_count_1,duration_ns_1,is_safe_2,automaton_depth_2,visited_count_2,duration_ns_2,is_safe,duration_ns
0,[],['hi-over-demand'],EDFVD (AC),edfvd,"['none', 'none', 'acbfs']",outputs/20260525_153517-fixed-quarter-clairvoy...,5,0.5,0.496078,5.0,...,True,0,0,0.000000e+00,True,14,4376,17514151,True,17514151
1,[],['hi-over-demand'],EDFVD (AC),edfvd,"['none', 'none', 'acbfs']",outputs/20260525_153517-fixed-quarter-clairvoy...,0,0.5,0.503846,5.0,...,True,0,0,0.000000e+00,True,11,715,4477057,True,4477057
2,[],['hi-over-demand'],EDFVD (AC),edfvd,"['none', 'none', 'acbfs']",outputs/20260525_153517-fixed-quarter-clairvoy...,6,0.5,0.503242,5.0,...,True,0,0,0.000000e+00,True,22,32816,123465089,True,123465089
3,[],['hi-over-demand'],EDFVD (AC),edfvd,"['none', 'none', 'acbfs']",outputs/20260525_153517-fixed-quarter-clairvoy...,1,0.5,0.499943,5.0,...,True,0,0,0.000000e+00,True,24,21112,115981596,True,115981596
4,[],['hi-over-demand'],EDFVD (AC),edfvd,"['none', 'none', 'acbfs']",outputs/20260525_153517-fixed-quarter-clairvoy...,7,0.5,0.499176,5.0,...,True,0,0,0.000000e+00,True,15,2536,9818780,True,9818780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131995,[],['hi-over-demand'],LWLF (PAC-P-AC),lwlf,"['pacbfs', 'pbfs', 'acbfs']",outputs/20260525_153517-fixed-quarter-clairvoy...,10917,1.0,0.997495,5.0,...,True,145629,1225273,1.565157e+09,True,209,4975803,57756749315,True,59324884902
131996,[],['hi-over-demand'],LWLF (PAC-P-AC),lwlf,"['pacbfs', 'pbfs', 'acbfs']",outputs/20260525_153517-fixed-quarter-clairvoy...,10627,1.0,0.997302,5.0,...,True,37825,780861,1.006557e+09,True,424,7507080,99144758717,True,100162937307
131997,[],['hi-over-demand'],LWLF (PAC-P-AC),lwlf,"['pacbfs', 'pbfs', 'acbfs']",outputs/20260525_153517-fixed-quarter-clairvoy...,10559,1.0,0.997050,5.0,...,True,280141,3384680,5.087329e+09,True,299,8051205,111600724636,True,116694913817
131998,[],['hi-over-demand'],LWLF (PAC-P-AC),lwlf,"['pacbfs', 'pbfs', 'acbfs']",outputs/20260525_153517-fixed-quarter-clairvoy...,10805,1.0,0.997471,5.0,...,True,12852,138419,2.052171e+08,True,187,9056754,101359628462,True,101569097410


Taskset Parameters:
- safe_oracles: safe oracles applied to the search algorithm
- unsafe_oracles: unsafe oracles applied to the search algorithm
- use_case: name of the scheduler and chained search algorithms configuration
- taskset_position: position of the taskset in the original generation

Split the raw table into:
- df_edfvd_test: from EDFVD_test column of the raw table, rename the use_case into "EDF-VD (sufficient)"
- df_stages[i]: from stage i of search algorithm of the raw table.
    - stage: i
    - search_algorithm: from search_algorithms[i]
    - schedulable: from is_safe_{i}
    - automaton_depth: from automaton_depth_{i}
    - visited_count: from visited_count_{i}
    - duration_ns: from duration_ns_{i}
- df_combined: for the combined final result of the search algorithm combinations.

In [37]:
# EDF-VD test
df_edfvd_test = df_sch[["taskset_position", "EDFVD_test", "U"]]
df_edfvd_test = df_edfvd_test.rename(columns={"EDFVD_test": "EDF-VD (sufficient)"})
df_edfvd_test["EDF-VD (sufficient)"] = df_edfvd_test["EDF-VD (sufficient)"].astype(bool)
df_edfvd_test = df_edfvd_test.melt(
    id_vars=["taskset_position", "U"], var_name="use_case", value_name="schedulable"
)
df_edfvd_test["stage"] = -1

df_edfvd_test

,taskset_position,U,use_case,schedulable,stage
0,5,0.5,EDF-VD (sufficient),True,-1
1,0,0.5,EDF-VD (sufficient),True,-1
2,6,0.5,EDF-VD (sufficient),True,-1
3,1,0.5,EDF-VD (sufficient),True,-1
4,7,0.5,EDF-VD (sufficient),True,-1
...,...,...,...,...,...
131995,10917,1.0,EDF-VD (sufficient),False,-1
131996,10627,1.0,EDF-VD (sufficient),False,-1
131997,10559,1.0,EDF-VD (sufficient),False,-1
131998,10805,1.0,EDF-VD (sufficient),False,-1


In [38]:
# new EDF-VD test
df_edfvd_test_new = df_sch[["taskset_position", "EDFVD_test_new", "U"]]
df_edfvd_test_new = df_edfvd_test_new.rename(columns={"EDFVD_test_new": "EDF-VD (sufficient, new)"})
df_edfvd_test_new["EDF-VD (sufficient, new)"] = df_edfvd_test_new["EDF-VD (sufficient, new)"].astype(bool)
df_edfvd_test_new = df_edfvd_test_new.melt(
    id_vars=["taskset_position", "U"], var_name="use_case", value_name="schedulable"
)
df_edfvd_test_new["stage"] = -1

df_edfvd_test_new

,taskset_position,U,use_case,schedulable,stage
0,5,0.5,"EDF-VD (sufficient, new)",True,-1
1,0,0.5,"EDF-VD (sufficient, new)",True,-1
2,6,0.5,"EDF-VD (sufficient, new)",True,-1
3,1,0.5,"EDF-VD (sufficient, new)",True,-1
4,7,0.5,"EDF-VD (sufficient, new)",True,-1
...,...,...,...,...,...
131995,10917,1.0,"EDF-VD (sufficient, new)",False,-1
131996,10627,1.0,"EDF-VD (sufficient, new)",False,-1
131997,10559,1.0,"EDF-VD (sufficient, new)",False,-1
131998,10805,1.0,"EDF-VD (sufficient, new)",False,-1


In [39]:
# EDF-VDSD test
df_edfvdsd_test = df_sch[["taskset_position", "EDFVDSD_test", "U"]]
df_edfvdsd_test = df_edfvdsd_test.rename(columns={"EDFVDSD_test": "EDF-VDSD"})
df_edfvdsd_test["EDF-VDSD"] = df_edfvdsd_test["EDF-VDSD"].astype(bool)
df_edfvdsd_test = df_edfvdsd_test.melt(
    id_vars=["taskset_position", "U"], var_name="use_case", value_name="schedulable"
)
df_edfvdsd_test["stage"] = -1

df_edfvdsd_test

,taskset_position,U,use_case,schedulable,stage
0,5,0.5,EDF-VDSD,True,-1
1,0,0.5,EDF-VDSD,True,-1
2,6,0.5,EDF-VDSD,True,-1
3,1,0.5,EDF-VDSD,True,-1
4,7,0.5,EDF-VDSD,True,-1
...,...,...,...,...,...
131995,10917,1.0,EDF-VDSD,False,-1
131996,10627,1.0,EDF-VDSD,False,-1
131997,10559,1.0,EDF-VDSD,False,-1
131998,10805,1.0,EDF-VDSD,False,-1


In [15]:
# Stages
n_stages = df_sch["search_algorithms"][0].count(",") + 1
df_stages = []

for i in range(n_stages):
    df_stage = df_sch[["taskset_position", "use_case", "U", f"is_safe_{i}", f"automaton_depth_{i}", f"visited_count_{i}", f"duration_ns_{i}"]]
    df_stage = df_stage.rename(columns={f"is_safe_{i}": "schedulable", f"automaton_depth_{i}": "automaton_depth", f"visited_count_{i}": "visited_count", f"duration_ns_{i}": "duration_ns"})
    df_stage["schedulable"] = df_stage["schedulable"].astype(bool)
    df_stage["stage"] = i
    df_stage["search_algorithm"] = df_sch["search_algorithms"].apply(lambda x: x.strip("[]").split(", ")[i].strip("'"))
    df_stages.append(df_stage)

df_stages[-1]

,taskset_position,use_case,U,schedulable,automaton_depth,visited_count,duration_ns,stage,search_algorithm
0,0,EDFVD (AC),0.5,True,34,35489,142300632,2,acbfs
1,3,EDFVD (AC),0.5,True,8,9762,35542128,2,acbfs
2,4,EDFVD (AC),0.5,True,19,10485,49724156,2,acbfs
3,6,EDFVD (AC),0.5,True,13,4801,17111192,2,acbfs
4,9,EDFVD (AC),0.5,True,13,2280,14458406,2,acbfs
...,...,...,...,...,...,...,...,...,...
263995,10934,LWLF (AC),1.0,True,503,3965714,51243936113,2,acbfs
263996,10803,LWLF (AC),1.0,False,92,8088699,98438533560,2,acbfs
263997,10917,LWLF (AC),1.0,True,209,4975803,60528114969,2,acbfs
263998,10805,LWLF (AC),1.0,True,187,9056754,105123029649,2,acbfs


In [40]:
df_total = df_sch[["taskset_position", "use_case", "U", "is_safe", "duration_ns"]]
df_total = df_total.rename(columns={"is_safe": "schedulable"})
df_total["schedulable"] = df_total["schedulable"].astype(bool)
df_total["stage"] = -1

df_total

,taskset_position,use_case,U,schedulable,duration_ns,stage
0,5,EDFVD (AC),0.5,True,17514151,-1
1,0,EDFVD (AC),0.5,True,4477057,-1
2,6,EDFVD (AC),0.5,True,123465089,-1
3,1,EDFVD (AC),0.5,True,115981596,-1
4,7,EDFVD (AC),0.5,True,9818780,-1
...,...,...,...,...,...,...
131995,10917,LWLF (PAC-P-AC),1.0,True,59324884902,-1
131996,10627,LWLF (PAC-P-AC),1.0,True,100162937307,-1
131997,10559,LWLF (PAC-P-AC),1.0,True,116694913817,-1
131998,10805,LWLF (PAC-P-AC),1.0,True,101569097410,-1


In [41]:
df_sched_comb = pd.concat([df_edfvd_test, df_edfvd_test_new, df_edfvdsd_test, df_total], ignore_index=True)
# df_sched_comb = pd.concat([df_edfvd_test, df_edfvd_test_new, df_total], ignore_index=True)
df_sched_comb

,taskset_position,U,use_case,schedulable,stage,duration_ns
0,5,0.5,EDF-VD (sufficient),True,-1,NaN
1,0,0.5,EDF-VD (sufficient),True,-1,NaN
2,6,0.5,EDF-VD (sufficient),True,-1,NaN
3,1,0.5,EDF-VD (sufficient),True,-1,NaN
4,7,0.5,EDF-VD (sufficient),True,-1,NaN
...,...,...,...,...,...,...
527995,10917,1.0,LWLF (PAC-P-AC),True,-1,5.932488e+10
527996,10627,1.0,LWLF (PAC-P-AC),True,-1,1.001629e+11
527997,10559,1.0,LWLF (PAC-P-AC),True,-1,1.166949e+11
527998,10805,1.0,LWLF (PAC-P-AC),True,-1,1.015691e+11


In [42]:
fig_size = (5, 5)

In [43]:
f, ax = plt.subplots(figsize=fig_size)

col_order = df_sched_comb["use_case"].unique()

sns.lineplot(
    data=df_sched_comb,
    x="U",
    y="schedulable",
    hue="use_case",
    ax=ax,
    markers=True,
    errorbar=None,
    style="use_case",
    ms=9,
    hue_order=col_order,
    style_order=col_order,
)

ax.set_ylabel("Schedulability ratio")
ax.set_xlabel("Average utilisation ($U^*$)")

# f.suptitle(
#     r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, T^{\mathsf{max}} = 30, n=5$",
#     x=0.55,
#     # y=0.7,
# )

ax.set_title(
    r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, T^{\mathsf{max}} = 30, n=5$",
)

# handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles=handles, labels=labels, framealpha=1, loc="lower left")
# ax.legend(
#     handles=handles,
#     labels=labels,
#     loc="upper center",
#     ncol=2,
#     framealpha=0,
#     columnspacing=0.8,
#     bbox_to_anchor=(0.5, 1.4),
# )
sns.move_legend(
    ax,
    loc="lower left",
    # ncol=2,
    framealpha=1,
    # columnspacing=0.8,
    # bbox_to_anchor=(0.5, 1.8),
    title="",
)

f.tight_layout()

/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_data = grouped.apply(agg, other).reset_index()
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
 

In [44]:
f.show()

In [11]:
date = datetime.datetime.now().strftime("%Y%m%d")
os.makedirs(f"./figs/{date}", exist_ok=True)
f.savefig(
    f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_scheduling.pdf",
    bbox_inches="tight",
)
f.savefig(
    f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_scheduling.png",
    bbox_inches="tight",
)
f.savefig(
    f"./figs/scheduling.pdf",
    bbox_inches="tight",
)

Plots the time taken vs average utilization for each algorithm

In [13]:
f, ax = plt.subplots(figsize=fig_size)

df_time = (
    df_sch.loc[:, ["U", "duration_ns", "use_case"]]
    .dropna(how="any")
    .assign(duration_s=lambda d: d["duration_ns"] / 1e9)
    .sort_values(["use_case", "U"])
)

sns.lineplot(
    data=df_time,
    x="U",
    y="duration_s",
    hue="use_case",
    style="use_case",
    marker="o",
    ax=ax,
)

ax.set_ylabel("Time taken (s)")
ax.set_xlabel("Average utilisation ($U^*$)")

ax.set_title(
    r"$P_{\mathsf{HI}} = 0.5, n=5$",
)

sns.move_legend(
    ax,
    loc="upper left",
    framealpha=1,
    title="",
)


/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_data = grouped.apply(agg, other).reset_index()
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
 

In [14]:
date = datetime.datetime.now().strftime("%Y%m%d")
os.makedirs(f"./figs/{date}", exist_ok=True)
f.savefig(
    f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_time_uavg.pdf",
    bbox_inches="tight",
)
f.savefig(
    f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_time_uavg.png",
    bbox_inches="tight",
)
f.savefig(
    "./figs/time_uavg.pdf",
    bbox_inches="tight",
)

Plot speed up compared to basedline ACBFS

In [ ]:
baseline_use_case = "EDF-VD (AC)"
df_baseline = df_sch[df_sch["use_case"] == baseline_use_case]

use_cases = df_sch["use_case"].unique()
test_cases = use_cases[use_cases != baseline_use_case]
test_cases

In [ ]:
def get_speedup(use_case):
    df_baseline = df_sch[df_sch["use_case"] == baseline_use_case]
    df_test = df_sch[df_sch["use_case"] == use_case]
    df_speedup = df_test.copy()
    df_speedup["speedup"] = df_baseline["duration_ns"] / df_test["duration_ns"]
    return df_speedup

In [ ]:
df_use_cases = {use_case: df_sch[df_sch["use_case"] == use_case].loc[:, ["U", "duration_ns", "is_safe", "use_case"]].reset_index(drop=True) for use_case in use_cases}
df_use_cases[baseline_use_case]

In [ ]:
rows = []

base_df = df_use_cases[baseline_use_case].reset_index(drop=True)

for test_case in test_cases:
    test_df = df_use_cases[test_case].reset_index(drop=True)

    tmp = test_df.copy()

    tmp["speedup"] = (
        base_df["duration_ns"] / test_df["duration_ns"]
    )

    tmp["comparison_case"] = test_case

    rows.append(
        tmp.loc[:, ["U", "speedup", "comparison_case", "is_safe"]]
    )

df_speedup_all = pd.concat(rows, ignore_index=True)

f, ax = plt.subplots(figsize=fig_size)

sns.boxplot(
    data=df_speedup_all,
    x="U",
    y="speedup",
    hue="comparison_case",
    dodge=True,
    ax=ax,
)

ax.set_yscale("log")

ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1,
)

ax.set_ylabel("Speedup vs baseline")
ax.set_xlabel(r"Average utilisation ($U^*$)")

ax.set_title(
    f"Speedup against {baseline_use_case}"
)

sns.move_legend(
    ax,
    loc="upper left",
    title="Test case",
    framealpha=1,
)

In [ ]:
plt.show()
date = datetime.datetime.now().strftime('%Y%m%d')
os.makedirs(f"./figs/{date}", exist_ok=True)
f.savefig(
    f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_speedup.pdf",
    bbox_inches="tight",
)
f.savefig(
    f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_speedup.png",
    bbox_inches="tight",
)
f.savefig(
    f"./figs/speedup.pdf",
    bbox_inches="tight",
)

Plot schedulability by stage for a specific algorithm

In [16]:
def plot_schedulability_by_stage(use_case):
    f, ax = plt.subplots(figsize=fig_size)

    df_scheds = []

    df_sched = df_sched_comb.loc[df_sched_comb["use_case"] == use_case]
    df_sched["search_algorithm"] = "Overall"
    df_scheds.append(df_sched)

    for i in range(n_stages):
        df_scheds.append(
            df_stages[i].loc[:, ["U", "schedulable", "use_case", "search_algorithm", "stage"]][df_stages[i]["use_case"] == use_case]
        )

    df_combined = pd.concat(df_scheds, ignore_index=True)
    search_algorithms = df_combined["search_algorithm"].unique()

    sns.lineplot(
        data=df_combined,
        x="U",
        y="schedulable",
        ax=ax,
        markers=True,
        errorbar=None,
        ms=9,
        hue="search_algorithm",
        hue_order=search_algorithms,
        style="search_algorithm",
        style_order=search_algorithms,
    )

    ax.set_ylabel("Schedulability ratio")
    ax.set_xlabel("Average utilisation ($U^*$)")
    ax.set_title(f'Schedulability by stages - {use_case}')

    return f

In [17]:
use_cases = df_sch["use_case"].unique()
use_cases

array(['EDFVD (AC)', 'EDFVDSD (AC)', 'LWLF (P-AC)', 'EDFVDSD (P-AC)',
       'LWLF (PAC-AC)', 'EDFVD (P-AC)', 'EDFVDSD (PAC-AC)',
       'LWLF (PAC-P-AC)', 'EDFVD (PAC-AC)', 'EDFVDSD (PAC-P-AC)',
       'EDFVD (PAC-P-AC)', 'LWLF (AC)'], dtype=object)

In [18]:
for use_case in use_cases:
    f = plot_schedulability_by_stage(use_case)
    date = datetime.datetime.now().strftime('%Y%m%d')
    os.makedirs(f"./figs/{date}", exist_ok=True)
    f.savefig(
        f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_schedulability_by_stage_{use_case.replace(' ', '_')}.pdf",
        bbox_inches="tight",
    )
    f.savefig(
        f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_schedulability_by_stage_{use_case.replace(' ', '_')}.png",
        bbox_inches="tight",
    )
    f.savefig(
        f"./figs/schedulability_by_stage_{use_case.replace(' ', '_')}.pdf",
        bbox_inches="tight",
    )

/tmp/ipykernel_772207/1422219883.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sched["search_algorithm"] = "Overall"
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_data = grouped.apply(agg, other).reset_index()
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-p

Plots the time taken by stage for a specific algorithm

In [19]:
rows = []

for test_case in test_cases:
    test_df = df_use_cases[test_case].reset_index(drop=True)

    tmp = test_df.copy()

    tmp["speedup"] = (
        base_df["duration_ns"] / test_df["duration_ns"]
    )

    tmp["comparison_case"] = test_case

    rows.append(
        tmp.loc[:, ["U", "speedup", "comparison_case", "is_safe"]]
    )

df_speedup_all = pd.concat(rows, ignore_index=True)

# Compute statistics
df_stats = (
    df_speedup_all
    .groupby(["comparison_case", "U"])["speedup"]
    .agg(
        geometric_mean=lambda x: np.exp(np.mean(np.log(x))),
        max_speedup="max",
    )
    .reset_index()
)
df_stats

NameError: name 'test_cases' is not defined

In [ ]:
f, ax = plt.subplots(figsize=fig_size)

sns.lineplot(
    data=df_stats,
    x="U",
    y="geometric_mean",
    hue="comparison_case",
    marker="o",
    ax=ax,
)

ax.set_ylabel("Geometric Mean Speedup")
ax.set_xlabel("U")

plt.show()

In [29]:
def plot_time_by_stage(use_case):
    f, ax = plt.subplots(figsize=fig_size)

    df_times = []
    for i in range(n_stages):
        df_stage = df_stages[i].loc[:, ["U", "duration_ns", "use_case", "search_algorithm", "stage"]]
        df_stage = df_stage[df_stage["use_case"] == use_case].dropna(how="any")
        df_times.append(
            df_stage.assign(
                duration_s=lambda d: d["duration_ns"] / 1e9,
            )
        )

    df_combined = pd.concat(df_times, ignore_index=True)
    df_combined = df_combined[df_combined["stage"].notna() & (df_combined["stage"] >= 0)]
    df_combined["stage"] = df_combined["stage"].astype(int)

    df_plot = (
        df_combined.groupby(["U", "stage"], as_index=False)["duration_s"]
        .mean()
        .sort_values(["U", "stage"])
    )

    stage_order = sorted(df_plot["stage"].unique())
    pivot = (
        df_plot.pivot(index="U", columns="stage", values="duration_s")
        .reindex(columns=stage_order)
        .fillna(0)
        .sort_index()
    )

    stage_to_search_alg = df_combined.set_index("stage")["search_algorithm"].to_dict()

    x = pivot.index.to_numpy(dtype=float)
    bottom = np.zeros(len(pivot), dtype=float)
    stage_colors = sns.color_palette("deep", n_colors=len(stage_order))

    for stage, color in zip(stage_order, stage_colors):
        values = pivot[stage].to_numpy(dtype=float)
        ax.bar(
            x,
            values,
            bottom=bottom,
            width=0.04,
            color=color,
            edgecolor="white",
            linewidth=0.4,
            label=f"Stage {stage + 1}: {stage_to_search_alg.get(stage, 'Unknown')}",
        )
        bottom += values

    ax.set_ylabel("Time taken (s)")
    ax.set_xlabel("Average utilisation ($U^*$)")
    ax.set_title(f"Time by stages - {use_case}")
    ax.legend(title="Stage", framealpha=1)

    return f

In [30]:
f = plot_time_by_stage(use_cases[1])
f.show()

In [31]:
for use_case in use_cases:
    f = plot_time_by_stage(use_case)
    date = datetime.datetime.now().strftime('%Y%m%d')
    os.makedirs(f"./figs/{date}", exist_ok=True)
    f.savefig(
        f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_time_by_stage_{use_case.replace(' ', '_')}.pdf",
        bbox_inches="tight",
    )
    f.savefig(
        f"./figs/{date}/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_time_by_stage_{use_case.replace(' ', '_')}.png",
        bbox_inches="tight",
    )
    f.savefig(
        f"./figs/time_by_stage_{use_case.replace(' ', '_')}.pdf",
        bbox_inches="tight",
    )